In [1]:
import os
from tqdm import tqdm
import pandas as pd
import subprocess
import pickle
import ast
import numpy as np
import matplotlib.pyplot as plt
#import pyts.image as pti
from collections import Counter

import pandas as pd
#import rpy2.robjects as robjects
#from rpy2.robjects import pandas2ri

import math
from sklearn.metrics import r2_score
from scipy.interpolate import interp1d
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

from specio import specread
from specio import help
from specio.datasets import load_sp_path

import pandas as pd
import os
from numpy.typing import NDArray
from scipy.signal import savgol_filter

In [ ]:
#INSTALLING PACKAGES

In [16]:
import os
import glob
import rpy2.robjects as ro

# Папка, куда устанавливаем R-пакеты
local_r_lib = os.path.expanduser("~/Rlibs")

# Папка с локальными архивами .tar.gz
r_packages_dir = os.path.abspath("R_packages")

print("Current working directory:")
print(os.getcwd())

print("\nR packages directory:")
print(r_packages_dir)

if not os.path.isdir(r_packages_dir):
    raise FileNotFoundError(f"Папка R_packages не найдена:\n{r_packages_dir}")

# ВАЖНО: порядок установки
packages_order = [
    "SparseM",
    "quadprog",
    "lpSolve",
    "limSolve",
    "baseline",
]

package_files = {}

for pkg in packages_order:
    matches = sorted(glob.glob(os.path.join(r_packages_dir, f"{pkg}_*.tar.gz")))
    
    if len(matches) == 0:
        raise FileNotFoundError(
            f"\nНе найден архив для пакета {pkg} в папке R_packages.\n"
            f"Ожидался файл вида: {pkg}_*.tar.gz\n\n"
            f"Сейчас в папке R_packages есть:\n"
            + "\n".join(os.listdir(r_packages_dir))
        )
    
    package_files[pkg] = matches[-1]

print("\nFound package archives:")
for pkg, path in package_files.items():
    print(f"{pkg}: {path}")

# Передаем пути в R
ro.globalenv["local_r_lib"] = local_r_lib
ro.globalenv["package_names"] = ro.StrVector(packages_order)
ro.globalenv["package_paths"] = ro.StrVector([package_files[pkg] for pkg in packages_order])

r_code = r'''
dir.create(local_r_lib, showWarnings = FALSE, recursive = TRUE)
.libPaths(c(local_r_lib, .libPaths()))

cat("\nR version:\n")
print(R.version.string)

cat("\nR library paths:\n")
print(.libPaths())

# Удаляем lock-файлы от неудачных установок
lock_dirs <- list.files(local_r_lib, pattern = "^00LOCK", full.names = TRUE)
if (length(lock_dirs) > 0) {
    cat("\nRemoving old lock directories:\n")
    print(lock_dirs)
    unlink(lock_dirs, recursive = TRUE, force = TRUE)
}

install_local_package <- function(pkg_name, pkg_file) {
    cat("\n----------------------------------------\n")
    cat("Installing package:", pkg_name, "\n")
    cat("From file:", pkg_file, "\n")
    
    if (!file.exists(pkg_file)) {
        stop(paste("File not found:", pkg_file))
    }
    
    # Удаляем старую/битую установку этого пакета
    pkg_dir <- file.path(local_r_lib, pkg_name)
    if (dir.exists(pkg_dir)) {
        cat("Removing existing package directory:", pkg_dir, "\n")
        unlink(pkg_dir, recursive = TRUE, force = TRUE)
    }
    
    install.packages(
        pkg_file,
        lib = local_r_lib,
        repos = NULL,
        type = "source",
        INSTALL_opts = c("--no-lock")
    )
    
    if (!requireNamespace(pkg_name, quietly = TRUE, lib.loc = local_r_lib)) {
        stop(paste("Package was not installed correctly:", pkg_name))
    }
    
    cat("Successfully installed:", pkg_name, "\n")
}

# Установка всех пакетов в правильном порядке
for (i in seq_along(package_names)) {
    install_local_package(package_names[[i]], package_paths[[i]])
}

cat("\n----------------------------------------\n")
cat("Trying to load baseline...\n")

library(baseline, lib.loc = local_r_lib)

cat("\nPackage baseline installed and loaded successfully!\n")
'''

ro.r(r_code)

Current working directory:
/home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction

R packages directory:
/home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction/R_packages

Found package archives:
SparseM: /home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction/R_packages/SparseM_1.84-2.tar.gz
quadprog: /home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction/R_packages/quadprog_1.5-8.tar.gz
lpSolve: /home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction/R_packages/lpSolve_5.6.23.tar.gz
limSolve: /home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction/R_packages/limSolve_1.5.7.1.tar.gz
baseline: /home/igor/WORK/_MP_ML_models/FINAL_DATABASE/4_baseline_correction/R_packages/baseline_1.3-5.tar.gz

R version:
[1] "R version 4.5.2 (2025-10-31)"

R library paths:
[1] "/home/igor/Rlibs"              "/usr/local/lib/R/site-library"
[3] "/usr/lib/R/site-library"       "/usr/lib/R/library"           

----------------------------

* installing *source* package ‘SparseM’ ...
** this is package ‘SparseM’ version ‘1.84-2’
** пакет ‘SparseM’ удачно распакован, MD5 sums проверены
ступенчатая инсталляция возможна только с блокировкой
** using non-staged installation
** libs
using C compiler: ‘gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0’
using Fortran compiler: ‘GNU Fortran (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0’


gfortran  -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong  -c bckslv.f -o bckslv.o
gfortran  -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong  -c chol.f -o chol.o
gfortran  -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong  -c chol2csr.f -o chol2csr.o
gfortran  -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong  -c cholesky.f -o cholesky.o
gfortran  -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong  -c csr.f -o csr.o
gfortran  -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong  -c extract.f -o extract.o
gcc -I"/usr/share/R/include" -DNDEBUG       -fpic  -g -O2 -fdebug-prefix-map=/build/r-base-5KMO92/r-base-4.5.2=. -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2  -c init.c -o init.o
gf

installing to /home/igor/Rlibs/SparseM/libs
** R
** data
** demo
** inst
** byte-compile and prepare package for lazy loading


Создание новой общей функции для ‘diag’ из пакета ‘base’ в package ‘SparseM’
Создание новой общей функции для ‘diag<-’ из пакета ‘base’ в package ‘SparseM’
Создание новой общей функции для ‘norm’ из пакета ‘base’ в package ‘SparseM’
Создание новой общей функции для ‘backsolve’ из пакета ‘base’ в package ‘SparseM’
Создание новой общей функции для ‘forwardsolve’ из пакета ‘base’ в package ‘SparseM’
Создание новой общей функции для ‘model.response’ из пакета ‘stats’ в package ‘SparseM’


...

Создание новой общей функции для ‘getCall’ в package ‘baseline’


** help
*** installing help indices
** building package indices
** testing if installed package can be loaded



----------------------------------------
Trying to load baseline...


* DONE (baseline)
R[write to console]: 
Присоединяю пакет: ‘baseline’


R[write to console]: Следующий объект скрыт от ‘package:stats’:

    getCall





Package baseline installed and loaded successfully!


In [ ]:
#LOADING PACKAGES

In [17]:
import rpy2.robjects as ro

ro.r('dir.create("~/Rlibs", showWarnings = FALSE, recursive = TRUE)')
ro.r('.libPaths(c("~/Rlibs", .libPaths()))')
ro.r('library(baseline)')

print("Package baseline loaded successfully!")

Package baseline loaded successfully!


In [ ]:
#MAIN

In [18]:
red_db_red = pd.read_csv("../2_compiling_unified_database_files/our_database_12_03_2025_red.csv")

In [19]:
red_db_red.head()

,File name,Color,Polymer,Matching,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,...,459.0,458.0,457.0,456.0,455.0,454.0,453.0,452.0,451.0,450.0
0,Adv1.1_3.csv,NaN,PE,0.98,0.001491,0.001483,0.001483,0.001492,0.001509,0.001532,...,0.023790,0.027825,0.032067,0.032876,0.030447,0.027955,0.027342,0.028074,0.028446,0.027935
1,Adv1.1_4.csv,white,PE,0.95,0.002901,0.002914,0.002915,0.002900,0.002873,0.002843,...,0.056986,0.056147,0.054138,0.051779,0.050458,0.050193,0.049825,0.048727,0.047823,0.047667
2,Adv1.1_5.csv,transparent,PE,0.92,0.002795,0.002797,0.002800,0.002804,0.002808,0.002809,...,0.097227,0.100712,0.101276,0.097796,0.091534,0.085392,0.081877,0.082037,0.083644,0.084670
3,Adv1.1_6.csv,blue,PP,0.96,0.001478,0.001477,0.001489,0.001509,0.001533,0.001553,...,0.018965,0.019365,0.019543,0.018910,0.018044,0.017325,0.016312,0.014865,0.013980,0.013988
4,Adv1.1_7.csv,black,PE,0.98,0.008695,0.008685,0.008666,0.008636,0.008603,0.008585,...,0.043483,0.044460,0.044129,0.043891,0.045927,0.049857,0.052803,0.052295,0.049565,0.047077


In [20]:
red_db_red_left = red_db_red[['File name', 'Color', 'Polymer', 'Matching']]
red_db_red_right = red_db_red.drop(['File name', 'Color', 'Polymer', 'Matching'], axis=1)

spectral_columns = list(red_db_red_right.columns)
x = np.array([float(i) for i in spectral_columns]) 

# Process each row in the DataFrame
fouling_indices = []
for idx, row in tqdm(red_db_red_right.iterrows()):
    y = np.array(list(row))     # Spectral data

    r_spectral_data = FloatVector(y)
    r_matrix = r['matrix'](r_spectral_data, nrow=1, ncol=len(r_spectral_data))
    fourS = ro.r('baseline.fillPeaks')
    fourS_result = fourS(r_matrix, 4, 25, 10, 355)
    corrected_data_4s = [max(0, x) for x in list(fourS_result[1])] #list(fourS_result[1])
    baseline_data_4s = list(fourS_result[0])
    
    yy_reversed = corrected_data_4s
    assert not np.isnan(yy_reversed).any(), f"specy_prep produced NaN: {yy_reversed}"

    red_db_red_right.loc[idx] = corrected_data_4s

2010it [03:21,  9.99it/s]


In [22]:
red_db_red_baseline = pd.concat([red_db_red_left, red_db_red_right], axis=1)
red_db_red_baseline.head()

,File name,Color,Polymer,Matching,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,...,459.0,458.0,457.0,456.0,455.0,454.0,453.0,452.0,451.0,450.0
0,Adv1.1_3.csv,NaN,PE,0.98,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000005,...,0.000000,0.000000,0.003756,0.004543,0.002093,0.000000,0.000000,0.000000,0.000005,0.000000
1,Adv1.1_4.csv,white,PE,0.95,0.000036,0.000051,0.000054,4.097968e-05,0.000015,0.000000,...,0.005088,0.004183,0.002108,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Adv1.1_5.csv,transparent,PE,0.92,0.000000,0.000000,0.000000,5.030889e-07,0.000007,0.000011,...,0.007167,0.010552,0.011016,0.007436,0.001075,0.000000,0.000000,0.000000,0.000000,0.000000
3,Adv1.1_6.csv,blue,PP,0.96,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000004,...,0.003438,0.003752,0.003843,0.003125,0.002173,0.001368,0.000269,0.000000,0.000000,0.000000
4,Adv1.1_7.csv,black,PE,0.98,0.000036,0.000028,0.000011,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.003556,0.006449,0.005889,0.003106,0.000566


In [23]:
red_db_red_baseline.to_csv('our_database_12_03_2025_red_with_FI.csv', index=False)